# mjlab で Unitree G1 の歩行を強化学習しよう（Colab・T4）

## このノートブックで行うこと
1. **mjlab を取得する**
2. **学習ログの設定（Weights & Biases）**
3. **強化学習（PPO）**：ヒューマノイド **Unitree G1** が、指示された速度（前後・左右・旋回）で歩けるように学習する
4. **デモ**：学習済みの方策（側転の動きをまねる）をダウンロードして，ブラウザのビューアで動かす
5. **自分で学習した方策を動かす**：3 で保存されたチェックポイントを読み込んで，ブラウザのビューアで確認する（6章）

ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選んでから実行してください．

## mjlab とは
- **Isaac Lab の書き方（manager-based API）** と，**MuJoCo Warp**（GPU で動く物理シミュレータ MuJoCo）を組み合わせた，ロボット学習のフレームワーク
- 何千台ものロボットを **GPU 上で同時にシミュレーション** して，強化学習を高速に回せる
- Isaac Lab（Isaac Sim 上で動く）は RTX の GPU と大きなメモリが必要で Colab では動かないが，mjlab は **Colab の T4 でも動く**
- スライドの「(Optional) Isaac Lab で Unitree G1 の歩行を強化学習」と同じ考え方（並列シミュレーション → PPO で歩行を学習）を，手軽に体験できる

## 強化学習（ここで学習するもの）のイメージ
| | 内容 |
|---|---|
| 観測 | 関節の角度・速度，体の傾き，指示された速度 など |
| 行動 | 各関節の目標角度 |
| 報酬 | 指示された速度に近い速さで歩けているか，転ばないか，無駄な動きをしていないか など |
| 学習アルゴリズム | PPO（rsl_rl） |

第5回の模倣学習（ACT）は「人のお手本をまねる」学習でした．強化学習は，お手本の代わりに **報酬** を手がかりに，試行錯誤しながら方策を良くしていきます．

## 1. mjlab の取得
GitHub から mjlab を取得して，そのフォルダに移動します．依存ライブラリは，このあと `uv run` を実行したときに自動でインストールされます（`uv` は Python のパッケージ管理ツール．Colab には最初から入っています）．

In [ ]:
!git clone https://github.com/mujocolab/mjlab.git
%cd mjlab

Cloning into 'mjlab'...
remote: Enumerating objects: 78056, done.
remote: Counting objects: 100% (6803/6803), done.
remote: Compressing objects: 100% (730/730), done.
remote: Total 78056 (delta 6691), reused 6074 (delta 6073), pack-reused 71253 (from 4)
Receiving objects: 100% (78056/78056), 93.86 MiB | 18.65 MiB/s, done.
Resolving deltas: 100% (61927/61927), done.
/content/mjlab


In [ ]:
import subprocess
import sys

## 2. 学習ログの設定（Weights & Biases）
mjlab は学習の経過（報酬など）を **wandb** に記録します．
- `wandb offline`：記録を Colab の中だけに保存する（**アカウント不要**）
- `wandb login`：アカウントがある人だけ．API キーを貼り付けると，後からブラウザでグラフを見られる（アカウントがなければ，この行は消して実行してよい）

In [ ]:
!wandb offline
!wandb login

wandb: Updated settings file /content/mjlab/wandb/settings
W&B offline. Running your script from this directory will only write metadata locally. Use `wandb disabled` to completely turn off W&B.
wandb: Loading settings from /content/mjlab/wandb/settings
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## 3. 強化学習（PPO）で G1 の速度追従歩行を学習する
- タスク `Mjlab-Velocity-Flat-Unitree-G1`：平らな地面の上で，指示された速度で歩く
- `--env.scene.num-envs 4096`：**4096台の G1 を GPU 上で同時に** 動かしてデータを集める（メモリ不足になったら 1024 などに減らす）
- 1回の更新（iteration）で「24ステップ × 4096台」の経験を集めて方策を更新する．最大 30,000 回
- 50回ごとに方策が `logs/rsl_rl/g1_velocity/<日時>/model_<回数>.pt` に保存される

⚠ このセルは学習が終わるまで止まりません（T4 だとかなり時間がかかります）．ログの報酬（reward）やエピソードの長さが増えていくのを確認したら，セル左の **■（停止）** で止めてかまいません．途中までの方策も保存されています．

In [ ]:
process = subprocess.Popen(
    [
        "uv",
        "run",
        "train",
        "Mjlab-Velocity-Flat-Unitree-G1",
        "--env.scene.num-envs",
        "4096",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

if process.stdout is None:
    raise RuntimeError("stdout could not be captured")

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv


## 4. デモ：学習済みの方策をブラウザで見る
- `uv run demo` は，mjlab が公開している **学習済みの方策（側転の動きをまねる：motion tracking）** をダウンロードして，8台の G1 を動かすビューアを起動します
- 3 で自分が学習した歩行の方策とは別のものです（「学習済みの方策を動かすとこうなる」という例）
- 実行するには，下のセルの先頭と末尾の `"""` を消してから実行してください（GPU を使うので，3 の学習は先に止めておく）
- 「Server is running!」と表示されたら，次の「OUTPUT」のセルを実行します

In [ ]:
"""
import subprocess
import sys

process = subprocess.Popen(
    [
        "uv",
        "run",
        "demo",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

if process.stdout is None:
    raise RuntimeError("stdout could not be captured")

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

    if "serving" in line.lower() or "running on" in line.lower() or "8081" in line:
        print("\n" + "=" * 50)
        print("✅ Server is running! Execute the next cell to view.")
        print("=" * 50)
        break
"""

## 5. ビューアを表示する
デモのサーバ（ポート 8081）を，Colab の画面の中に表示します．マウスで視点を動かせます．

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8081)

## 6. 自分で学習した方策を動かす
3 で保存された方策（チェックポイント）を読み込んで，ブラウザのビューアで G1 を動かします．

⚠ 3 の学習や 4 のデモが動いていたら，先に **■（停止）** で止めてください（どちらも GPU を使います）．

### 6-1. 保存された方策を確認する
50回ごとに `model_<回数>.pt` が保存されています（例：104回で止めたら `model_0.pt`・`model_50.pt`・`model_100.pt`）．

In [ ]:
!ls -t logs/rsl_rl/g1_velocity/*/ | head

### 6-2. 学習した方策でビューアを起動する
一番新しい学習の，一番新しいチェックポイントを自動で選びます．特定のものを使いたいときは `ckpt` にパスを直接書いてください．

In [ ]:
import glob, subprocess, sys

# 一番新しい学習の，一番新しいチェックポイントを使う
ckpt = sorted(glob.glob("logs/rsl_rl/g1_velocity/*/model_*.pt"),
              key=lambda p: (p.split("/")[-2], int(p.split("_")[-1][:-3])))[-1]
print("使うチェックポイント:", ckpt)

process = subprocess.Popen(
    ["uv", "run", "play", "Mjlab-Velocity-Flat-Unitree-G1",
     "--checkpoint-file", ckpt,
     "--num-envs", "8",
     "--viewer", "viser"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end=""); sys.stdout.flush()
    if "http://" in line or "8081" in line:
        print("\n✅ ビューアが起動しました．次のセルを実行してください．")
        break

### 6-3. ビューアを表示する
6-2 の出力に表示された URL のポート番号を確認してください．`8081` 以外（例：`8082`）なら，下の数字を合わせます．

- ビューアのチェックポイント切り替えの機能で，`model_0` → `model_50` → `model_100` … と切り替えると，学習が進むにつれて動きが変わる様子を比べられます
- 学習回数が少ないうち（数百回程度）は，まだ転んだりふらついたりします．上限は 30,000 回です

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8081)

## 参考
- mjlab（GitHub）：https://github.com/mujocolab/mjlab
- mjlab ドキュメント：https://mujocolab.github.io/mjlab/
- 論文：K. Zakka et al.（2026）"mjlab: A Lightweight Framework for GPU-Accelerated Robot Learning," arXiv:2601.22074
- Isaac Lab（GitHub）：https://github.com/isaac-sim/IsaacLab